# Model Hub - Importing models via H2O Drive

This notebook uses the H2O Drive Python Client (v4) to import a model downloaded from Hugging Face into H2O AI Cloud. Models written H2O Model Hub can be:
- Used across the H2O AI platform
- Shared with other users and services
- Operated on via some Hugging Face libraries

## Required permissions

As this notebook will guide us through uploading data to H2O AI Cloud, we must have the appropriate access permissions to do so.

Unless modified, this notebook will upload a model to the "global" H2O Model Hub registry, backed by the H2O Drive bucket for the "global" H2O workspace.

**Thus, please ensure you have the correct level of access to write to the "global" workspace.** Contact your H2O AI Cloud adminstrator for any questions.

## Helpers

In this section, we install packages and define helpers used in the rest of the notebook. This section is safe to skim over or to read at your own leisure.

In [ ]:
import sys

!{sys.executable} -m pip install -q "h2o-drive>=4.0.0"
!{sys.executable} -m pip install -q huggingface_hub

In [ ]:
import fnmatch
import os
from typing import List

import h2o_drive

_MODELHUB_BUCKET_PREFIX = ".modelhub/data/"

async def upload_folder(
        bucket: h2o_drive.Bucket,
        repo_id: str,
        folder_path: str,
        *,
        revision: str = "main",
        ignore_patterns: List[str] = [],
) -> None:
    # We expect the specified bucket to not be prefixed.
    # For convenience, we rebase to the prefix which Model Hub reads from.
    modelhub_bucket = bucket.with_prefix(_MODELHUB_BUCKET_PREFIX)

    for root, dirs, files in os.walk(folder_path):
        for file in files:
            # Compute file paths.
            full_filepath = os.path.join(root, file)
            relative_filepath = os.path.relpath(full_filepath, folder_path)

            # Skip file if it matches an ignored pattern.
            if any(fnmatch.fnmatch(relative_filepath, p) for p in ignore_patterns):
                continue

            # Upload to the Drive bucket under an appropriate key.
            key = f"{repo_id}/{revision}/{relative_filepath}"
            await modelhub_bucket.upload_file(full_filepath, key)

            # Log the upload.
            print(f"{relative_filepath} uploaded to Model Hub repo {repo_id}")

## Download a model from Hugging Face

In this section, we download the `albert/albert-base-v2` model from Hugging Face in preparation to then upload it to H2O AI Cloud.

> 💡 Tip
>
> This is just one example of how to source a Hugging Face repository. You may instead use any method of retrieving model files.
>
> See Hugging Face's how-to guide for information on other ways to download Hugging Face repository files:
> https://huggingface.co/docs/huggingface_hub/guides/download

Let's decide on where to temporarily download the model. Change this directory if necessary based on your environment.

In [ ]:
import os

current_directory = os.getcwd()

download_dir = os.path.join(current_directory, "downloaded_model")

We'll now use Hugging Face's `snapshot_download()` function to download the desired model repository files. Supposing that we want to ignore certain model formats, we'll also declare some file patterns to ignore.

For more information about `snapshot_download()` and available its options, see the [relevant Hugging Face docs](https://huggingface.co/docs/huggingface_hub/main/en/package_reference/file_download#huggingface_hub.snapshot_download).

In [ ]:
import huggingface_hub as hf

hf.snapshot_download(
    repo_id="albert/albert-base-v2",
    local_dir=download_dir,
    ignore_patterns=["*.msgpack", "*.h5", "*.ot"],
)

## Connect to H2O Drive

In this section, we connect to H2O Drive in preparation of uploading our model.

> 📢 Important
>
> This section assumes that an H2O AI Cloud environment can be discovered from your environment.
> On local environments, this means having the H2O CLI installed and configured.
>
> For information on connecting to Drive from different environments, see the notebook tutorial titled _"Drive - Connecting from different environments"_.

H2O Drive provides object storage for H2O AI Cloud. Objects in Drive can be used across the H2O AI platform and shared with other users and services.

In order to upload our model to Drive, we first need to connect to it.

In [ ]:
import h2o_drive

drive = h2o_drive.connect()

To upload a model to the "global" H2O Model Hub registry, we'll be uploading to the Drive bucket for the "global" H2O workspace.

Let's open that bucket now.

In [ ]:
bucket = drive.workspace_bucket("global")

## Upload model

With the model files downloaded, and a connection to H2O Drive open, we're ready to upload the model to H2O AI Cloud.

Using the `upload_folder()` helper function defined at the top of this notebook, we will:
- Upload the model files from the local `download_dir` directory we have them saved in.
- Upload the model files with the same repository ID, `albert/albert-base-v2`, as the original.
- Skip uploading files matching certain patterns (i.e., any caches that may have been created).

In [ ]:
repo_id = "albert/albert-base-v2"
ignore_patterns = [".cache*"]

await upload_folder(
    bucket=bucket,
    repo_id=repo_id,
    folder_path=download_dir,
    ignore_patterns=ignore_patterns,
)

🎉 That's it! The model is now uploaded to H2O Drive, where it can be used across the H2O AI platform and shared with users and services.

As a result, the model can now also be retrieved, and operated on, via some Hugging Face libraries while being stored in H2O AI Cloud. For examples, see the notebook tutorial titled _"Model Hub - Using Hugging Face libraries"_.

## Clean up

Let's clean up the temporary model files we downloaded.

In [ ]:
import shutil

shutil.rmtree(download_dir)